In [ ]:


import math
import heapq
import time
from dataclasses import dataclass
from typing import List, Tuple, Dict, Set, Optional
import matplotlib.pyplot as plt

# ============================================================
# CONSTANTS AND GEOMETRIC PRIMITIVES
# ============================================================

EPSILON = 1e-9

@dataclass(frozen=True)
class Point:
    """A point in the plane with Euclidean distance."""
    x: float
    y: float

    def distance_to(self, other: "Point") -> float:
        return math.hypot(self.x - other.x, self.y - other.y)


class Polygon:
    """Convex polygon defined by a list of vertices in counter-clockwise order."""

    def __init__(self, vertices: List[Point]):
        self.vertices = vertices
        self.edges = []
        n = len(vertices)
        for i in range(n):
            self.edges.append((vertices[i], vertices[(i + 1) % n]))


# ============================================================
# ROBUST GEOMETRIC PREDICATES
# ============================================================

def orientation(a: Point, b: Point, c: Point) -> int:
    """
    Returns:
      0 if points are collinear
      1 if clockwise
      -1 if counter‑clockwise
    """
    val = (b.y - a.y) * (c.x - b.x) - (b.x - a.x) * (c.y - b.y)
    if abs(val) < EPSILON:
        return 0
    return 1 if val > 0 else -1


def on_segment(a: Point, b: Point, c: Point) -> bool:
    """Checks if point b lies on the closed segment a‑c."""
    return (min(a.x, c.x) - EPSILON <= b.x <= max(a.x, c.x) + EPSILON and
            min(a.y, c.y) - EPSILON <= b.y <= max(a.y, c.y) + EPSILON)


def segments_intersect(p1: Point, q1: Point, p2: Point, q2: Point) -> bool:
    """
    Robust test whether segments p1‑q1 and p2‑q2 intersect (including endpoints).
    """
    o1 = orientation(p1, q1, p2)
    o2 = orientation(p1, q1, q2)
    o3 = orientation(p2, q2, p1)
    o4 = orientation(p2, q2, q1)

    # General case: proper intersection
    if o1 != o2 and o3 != o4:
        return True

    # Special cases: collinear endpoint overlaps
    if o1 == 0 and on_segment(p1, p2, q1): return True
    if o2 == 0 and on_segment(p1, q2, q1): return True
    if o3 == 0 and on_segment(p2, p1, q2): return True
    if o4 == 0 and on_segment(p2, q1, q2): return True

    return False


def point_inside_convex_polygon(point: Point, polygon: Polygon) -> bool:
    """
    Strict interior test for convex polygon.
    Returns True if point is strictly inside, False if on boundary or outside.
    """
    # Use winding number / cross product sign consistency
    sign = None
    for a, b in polygon.edges:
        cross = (b.x - a.x) * (point.y - a.y) - (b.y - a.y) * (point.x - a.x)
        if abs(cross) < EPSILON:
            continue                     # point on edge → not inside
        current = cross > 0
        if sign is None:
            sign = current
        elif sign != current:
            return False                 # mixed signs → outside
    return sign is not None


# ============================================================
# VISIBILITY GRAPH
# ============================================================

class VisibilityGraph:
    """
    Builds a graph where vertices are polygon corners plus start and goal.
    Edges connect vertices if the straight line segment does not intersect
    the interior of any obstacle.
    """

    def __init__(self, obstacles: List[Polygon], start: Point, goal: Point):
        self.obstacles = obstacles
        self.start = start
        self.goal = goal

        # Collect all vertices
        self.vertices = []
        for poly in obstacles:
            self.vertices.extend(poly.vertices)
        self.vertices.append(start)
        self.vertices.append(goal)

        # Remove duplicates (floating point equality)
        unique = {}
        for v in self.vertices:
            key = (round(v.x, 9), round(v.y, 9))
            unique[key] = v
        self.vertices = list(unique.values())

        # Map vertex to index for adjacency list
        self.index = {v: i for i, v in enumerate(self.vertices)}
        self.adj = [[] for _ in range(len(self.vertices))]

        self._build_graph()

    def _build_graph(self):
        n = len(self.vertices)
        for i in range(n):
            for j in range(i + 1, n):
                vi = self.vertices[i]
                vj = self.vertices[j]
                if self._is_visible(vi, vj):
                    dist = vi.distance_to(vj)
                    self.adj[i].append((j, dist))
                    self.adj[j].append((i, dist))

    def _is_visible(self, a: Point, b: Point) -> bool:
        """Returns True if the segment a‑b does not intersect any obstacle interior."""
        for poly in self.obstacles:
            # Check intersection with polygon edges
            for (e1, e2) in poly.edges:
                # Skip if the segment shares an endpoint with the edge (touching is allowed)
                if a == e1 or a == e2 or b == e1 or b == e2:
                    continue
                if segments_intersect(a, b, e1, e2):
                    return False

            # If the midpoint lies strictly inside the polygon, the segment cuts through
            mid = Point((a.x + b.x) * 0.5, (a.y + b.y) * 0.5)
            if point_inside_convex_polygon(mid, poly):
                return False
        return True


# ============================================================
# SEARCH PROBLEM INTERFACE
# ============================================================

class PathfindingProblem:
    """Formal search problem for graph search algorithms."""

    def __init__(self, graph: VisibilityGraph):
        self.graph = graph

    def get_start_state(self) -> Point:
        return self.graph.start

    def is_goal_state(self, state: Point) -> bool:
        return state == self.graph.goal

    def get_successors(self, state: Point):
        idx = self.graph.index[state]
        return [(self.graph.vertices[nei], cost) for nei, cost in self.graph.adj[idx]]


# ============================================================
# SEARCH ALGORITHMS
# ============================================================

def uniform_cost_search(problem: PathfindingProblem):
    """Uniform Cost Search (Dijkstra) on the visibility graph."""
    start = problem.get_start_state()
    frontier = [(0, start)]
    came_from = {}
    cost_so_far = {start: 0}
    nodes_expanded = 0
    start_time = time.perf_counter()

    while frontier:
        current_cost, current = heapq.heappop(frontier)
        nodes_expanded += 1

        if problem.is_goal_state(current):
            end_time = time.perf_counter()
            path = reconstruct_path(came_from, current)
            stats = {
                "cost": current_cost,
                "expanded": nodes_expanded,
                "runtime": end_time - start_time
            }
            return path, stats

        for neighbor, edge_cost in problem.get_successors(current):
            new_cost = current_cost + edge_cost
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                came_from[neighbor] = current
                heapq.heappush(frontier, (new_cost, neighbor))

    return None, None


def heuristic(state: Point, goal: Point) -> float:
    """Euclidean distance – admissible and consistent."""
    return state.distance_to(goal)


def a_star_search(problem: PathfindingProblem):
    """A* search with Euclidean heuristic."""
    start = problem.get_start_state()
    goal = problem.graph.goal
    frontier = [(0, start)]
    came_from = {}
    g_score = {start: 0}
    nodes_expanded = 0
    start_time = time.perf_counter()

    while frontier:
        _, current = heapq.heappop(frontier)
        nodes_expanded += 1

        if problem.is_goal_state(current):
            end_time = time.perf_counter()
            path = reconstruct_path(came_from, current)
            stats = {
                "cost": g_score[current],
                "expanded": nodes_expanded,
                "runtime": end_time - start_time
            }
            return path, stats

        for neighbor, edge_cost in problem.get_successors(current):
            tentative_g = g_score[current] + edge_cost
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                came_from[neighbor] = current
                priority = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(frontier, (priority, neighbor))

    return None, None


def reconstruct_path(came_from: dict, current: Point) -> List[Point]:
    """Reconstruct the path from start to current."""
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path


# ============================================================
# VISUALIZATION
# ============================================================

def visualize(obstacles: List[Polygon],
              graph: VisibilityGraph,
              path: List[Point],
              title: str = "Shortest Path (A*)"):
    """Plot polygons, visibility graph edges, and the solution path."""
    plt.figure(figsize=(10, 8))

    # Draw obstacles
    for poly in obstacles:
        xs = [p.x for p in poly.vertices]
        ys = [p.y for p in poly.vertices]
        xs.append(poly.vertices[0].x)
        ys.append(poly.vertices[0].y)
        plt.plot(xs, ys, 'k-', linewidth=2)

    # Draw visibility graph (light grey)
    for i in range(len(graph.vertices)):
        for j, _ in graph.adj[i]:
            if i < j:
                xi, yi = graph.vertices[i].x, graph.vertices[i].y
                xj, yj = graph.vertices[j].x, graph.vertices[j].y
                plt.plot([xi, xj], [yi, yj], color='lightgray', alpha=0.5)

    # Draw solution path
    if path:
        px = [p.x for p in path]
        py = [p.y for p in path]
        plt.plot(px, py, 'r-', linewidth=3, label='Shortest path')

    # Mark start and goal
    plt.plot(graph.start.x, graph.start.y, 'go', markersize=10, label='Start')
    plt.plot(graph.goal.x, graph.goal.y, 'ro', markersize=10, label='Goal')

    plt.title(title)
    plt.axis('equal')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()


# ============================================================
# BENCHMARK SCENARIOS
# ============================================================

def scenario_easy():
    """Single square obstacle."""
    obstacles = [Polygon([
        Point(3, 3), Point(3, 6), Point(6, 6), Point(6, 3)
    ])]
    start = Point(0, 0)
    goal = Point(10, 10)
    return obstacles, start, goal


def scenario_medium():
    """Two convex polygons (square and triangle)."""
    obstacles = [
        Polygon([Point(2, 2), Point(2, 5), Point(5, 5), Point(5, 2)]),
        Polygon([Point(7, 3), Point(9, 6), Point(11, 2)])
    ]
    start = Point(0, 0)
    goal = Point(13, 8)
    return obstacles, start, goal


def scenario_hard():
    """Three convex polygons with irregular placement."""
    obstacles = [
        Polygon([Point(1, 4), Point(2, 7), Point(5, 6), Point(4, 3)]),
        Polygon([Point(6, 1), Point(9, 2), Point(8, 5), Point(5, 4)]),
        Polygon([Point(10, 6), Point(12, 9), Point(14, 7), Point(13, 4)])
    ]
    start = Point(0, 0)
    goal = Point(15, 10)
    return obstacles, start, goal


# ============================================================
# EXPERIMENT DRIVER
# ============================================================

def run_experiment(name: str, scenario_fn):
    print("\n" + "=" * 60)
    print(f"SCENARIO: {name}")
    print("=" * 60)

    obstacles, start, goal = scenario_fn()
    graph = VisibilityGraph(obstacles, start, goal)
    problem = PathfindingProblem(graph)

    # UCS
    print("\nUniform Cost Search")
    ucs_path, ucs_stats = uniform_cost_search(problem)
    if ucs_stats:
        print(f"  Path cost: {ucs_stats['cost']:.4f}")
        print(f"  Nodes expanded: {ucs_stats['expanded']}")
        print(f"  Runtime: {ucs_stats['runtime']:.6f} s")
    else:
        print("  No path found.")

    # A*
    print("\nA* Search")
    astar_path, astar_stats = a_star_search(problem)
    if astar_stats:
        print(f"  Path cost: {astar_stats['cost']:.4f}")
        print(f"  Nodes expanded: {astar_stats['expanded']}")
        print(f"  Runtime: {astar_stats['runtime']:.6f} s")
    else:
        print("  No path found.")

    # Visualize using A* solution (if exists)
    if astar_path:
        visualize(obstacles, graph, astar_path, title=f"{name} - A* Solution")


if __name__ == "__main__":
    run_experiment("Easy", scenario_easy)
    run_experiment("Medium", scenario_medium)
    run_experiment("Hard", scenario_hard)